# Branch Sales Report

*Combine array creation, broadcasting, axis reductions, and Boolean selection in one sales report.*


## Problem Definition

### Question

Three branches sell four products at shared unit prices. What is the
revenue for every branch and product, and which branches reach
900,000 KRW in total revenue?

| Branch | Notebook | Pen | Folder | Planner |
|---|---:|---:|---:|---:|
| North | 120 | 80 | 95 | 110 |
| Central | 100 | 90 | 105 | 115 |
| South | 130 | 70 | 100 | 120 |
| Unit price (KRW) | 2000 | 1500 | 1200 | 3500 |

### Constraints

- Store unit counts and prices in separate integer arrays.
- Rows represent branches; columns represent products throughout.
- Use broadcasting for revenue and axis reductions for totals.
- Select qualifying branches with a Boolean mask.
- Define all data here and preserve the original unit counts.


## Session Method Map

| Guided topic | Role in this report |
|---|---|
| [Why NumPy?](01_why_numpy.ipynb) | Calculate with whole arrays |
| [Array creation](02_array_creation.ipynb) | Prepare counts and prices |
| [Array attributes](03_array_attributes.ipynb) | Confirm `(3, 4)` and `(4,)` shapes |
| [Indexing](04_indexing_and_slicing.ipynb) | Read the planner revenue column |
| [Broadcasting](05_operations_and_broadcasting.ipynb) | Apply prices to every branch |
| [Aggregation and filtering](06_aggregation_and_filtering.ipynb) | Summarize and select branches |


## Program Flow

```text
Prepared counts and prices -> revenue table -> totals and branch mask -> report
```


## Imports and Prepared Data

The names follow the same row and column order as the numeric arrays.
Keep them separate so each array has one clear role and data type.

**Question.** Will the branch with the largest unit count also have
the largest revenue in this dataset?


In [1]:
import numpy as np

branch_names = np.array(["North", "Central", "South"])
product_names = np.array(["Notebook", "Pen", "Folder", "Planner"])
sales = np.array([[120, 80, 95, 110],
                  [100, 90, 105, 115],
                  [130, 70, 100, 120]])
prices = np.array([2000, 1500, 1200, 3500])
revenue_target = 900000

print("Sales shape:", sales.shape)
print("Price shape:", prices.shape)
print("Unit totals by branch:", sales.sum(axis=1))


Sales shape: (3, 4)
Price shape: (4,)
Unit totals by branch: [405 410 420]


The shapes match three branches and four prices. Unit totals are
`[405, 410, 420]`, so South sells the most units. Revenue also depends
on which products each branch sells.


## Calculating the Revenue Table

Multiplication combines unit counts and product prices at matching
columns. The returned array stores revenue and leaves the counts intact.


In [2]:
def calculate_revenue(unit_counts, unit_prices):
    return unit_counts * unit_prices


## Summarizing and Selecting

`axis=1` produces branch totals; `axis=0` produces product totals.
The branch mask is built from revenue, so it has one Boolean per branch.


In [3]:
def summarize_revenue(revenue, target):
    branch_totals = revenue.sum(axis=1)
    product_totals = revenue.sum(axis=0)

    return {
        "branch_totals": branch_totals,
        "product_totals": product_totals,
        "grand_total": revenue.sum(),
        "qualifying_mask": branch_totals >= target,
    }


## Reporting the Results

The loops below only format the report. NumPy has already performed
the calculations. Column index 3 is the planner product in the prepared data.


In [4]:
def print_report(revenue, summary):
    print("Revenue table (KRW):")
    print(revenue)
    print("Planner revenue by branch:", revenue[:, 3])

    for name, total in zip(branch_names, summary["branch_totals"]):
        print(f"{name} revenue: {total:,} KRW")

    for name, total in zip(product_names, summary["product_totals"]):
        print(f"{name} total: {total:,} KRW")

    print(f"Grand total: {summary['grand_total']:,} KRW")
    print("Branches reaching the target:", branch_names[summary["qualifying_mask"]])


## Running the Workflow

`main()` connects the calculation, summary, and report. The final check
confirms that both ways of grouping revenue preserve the same grand total.


In [5]:
def main():
    revenue = calculate_revenue(sales, prices)
    summary = summarize_revenue(revenue, revenue_target)
    print_report(revenue, summary)
    print(
        "Totals agree:",
        summary["branch_totals"].sum()
        == summary["product_totals"].sum()
        == summary["grand_total"],
    )
    return revenue, summary


revenue, report = main()


Revenue table (KRW):
[[240000 120000 114000 385000]
 [200000 135000 126000 402500]
 [260000 105000 120000 420000]]
Planner revenue by branch: [385000 402500 420000]
North revenue: 859,000 KRW
Central revenue: 863,500 KRW
South revenue: 905,000 KRW
Notebook total: 700,000 KRW
Pen total: 360,000 KRW
Folder total: 360,000 KRW
Planner total: 1,207,500 KRW
Grand total: 2,627,500 KRW
Branches reaching the target: ['South']
Totals agree: True


## Result Analysis

Total revenue is **2,627,500 KRW**. North earns 859,000 KRW, Central
earns 863,500 KRW, and South earns 905,000 KRW. South is the only branch
that reaches the 900,000 KRW target.

The prediction is correct for these data: South has both the largest
unit count and the largest revenue. This need not hold for a different
product mix because prices differ.

Product totals are `[700000, 360000, 360000, 1207500]` in the listed
product order. Planners contribute the most revenue. `Totals agree`
is `True`, confirming that row and column reductions account for the
same twelve revenue cells.
